# Stage 1a — Local theory validation (CPU, seconds, no RL)

Report reference: `PROJECT_REPORT.md` §R3. This notebook is an **orchestration layer only** (decision D13) — every function it calls lives in `safelie.theory` / `safelie.consensus`, none is defined here.

Runs Test A (Theorem 1, open-loop mass conservation), Test B (Proposition 1, spreading and stealth), and Test C (the closed-loop synthetic diagnostic addressing weakness W1). All three are pure numerical checks of the paper's dual recursion — no environment, no policy, no critic.

In [ ]:
import numpy as np

from safelie.consensus.topologies import build_topology
from safelie.theory import run_closed_loop_diagnostic, verify_mass_conservation, verify_spreading

N_AGENTS = 6
ETA_LAMBDA = 0.035  # [SPEC]
K = 500

## Test A — Theorem 1, open-loop mass conservation

`1^T e_K = eta_lambda * sum_k 1^T delta_k`, independent of the mixing matrix W. Expected: exact agreement to 1e-10 across every topology below.

In [ ]:
topologies = {
    "complete": build_topology("complete", N_AGENTS),
    "ring": build_topology("ring", N_AGENTS),
    "star": build_topology("star", N_AGENTS),
    "erdos_renyi_p05": build_topology("erdos_renyi", N_AGENTS, p=0.5, graph_seed=1),
    "identity": build_topology("identity", N_AGENTS),
    "shared_constraint": build_topology("shared_constraint", N_AGENTS),
}

delta_vec = np.zeros(N_AGENTS)
delta_vec[0] = 1.5  # persistent single-agent corruption, agent 0
delta_schedule = [delta_vec.copy() for _ in range(K)]

result_a = verify_mass_conservation(topologies, delta_schedule, ETA_LAMBDA, tol=1e-10)
print("Per-topology aggregate dual bias (1^T e_K):")
for name, mass in result_a.masses.items():
    print(f"  {name:<20s} {mass:.10f}")
print(f"\nClosed-form expected: {result_a.expected:.10f}")
print(f"Max deviation from closed form: {result_a.max_abs_deviation:.3e}")
print(f"Cross-topology spread: {result_a.cross_topology_spread:.3e}")
print(f"TEST A: {'PASS' if result_a.passed else 'FAIL'}")

## Test B — Proposition 1, spreading and stealth

For a single corrupted agent, every agent's multiplier converges to the *same* bias under consensus (uniform spreading), versus concentrating entirely on the corrupted agent when `W = I`. The stealth claim: median-referenced deviation stays bounded under consensus, but grows linearly in K under `W = I`.

In [ ]:
W_ring = build_topology("ring", N_AGENTS)
W_identity = build_topology("identity", N_AGENTS)

result_ring = verify_spreading(W_ring, delta=2.0, j=0, n_agents=N_AGENTS, eta=ETA_LAMBDA, K=K)
result_identity = verify_spreading(W_identity, delta=2.0, j=0, n_agents=N_AGENTS, eta=ETA_LAMBDA, K=K)

print("Ring (connected consensus):")
print(f"  e_K = {np.round(result_ring.e_K, 4)}")
print(f"  residual norm = {result_ring.residual_norm:.4f} <= bound {result_ring.residual_bound:.4f}: {result_ring.residual_bound_satisfied}")
print(f"  median-referenced deviation = {result_ring.median_deviation:.4f}")

print("\nIdentity (non-communicating control):")
print(f"  e_K = {np.round(result_identity.e_K, 4)}")
print(f"  median-referenced deviation = {result_identity.median_deviation:.4f}")

print(f"\nStealth ratio (identity / ring): {result_identity.median_deviation / max(result_ring.median_deviation, 1e-9):.1f}x")

## Test C — Closed-loop synthetic diagnostic (addresses weakness W1)

Theorem 1's proof assumes both trajectories share the same primal sequence — an open-loop condition the real system violates. This diagnostic closes the loop with a per-agent linear feedback from the multiplier back onto the residual and measures `rho`, the relative spread of the aggregate bias across topologies. `rho ~ 0` means invariance survives; a large `rho` is a finding about the theorem's scope, not a bug — the gate requires this to *run and be recorded*, not to come out any particular way (PROJECT_REPORT.md §R3.4).

In [ ]:
rng = np.random.default_rng(0)
gains = rng.uniform(0.005, 0.05, size=N_AGENTS)  # heterogeneous per-agent feedback gain

result_c = run_closed_loop_diagnostic(
    topologies={"complete": topologies["complete"], "ring": topologies["ring"], "star": topologies["star"]},
    n_agents=N_AGENTS,
    K=K,
    eta=ETA_LAMBDA,
    c=gains,
    d=25.0,
    delta=2.0,
    corrupted_agent=0,
)

print(f"Mean feedback gain: {result_c.mean_gain:.4f}, heterogeneity (std): {result_c.gain_heterogeneity:.4f}")
print("Per-topology closed-loop aggregate bias (1^T e_K):")
for name, val in result_c.aggregate_bias.items():
    print(f"  {name:<12s} {val:.4f}")
print(f"\nOpen-loop reference (Theorem 1's exact value): {result_c.open_loop_reference:.4f}")
print(f"rho (relative cross-topology spread, closed loop): {result_c.rho:.4f}")

## Exit criterion (PROJECT_REPORT.md §R3.4)

- Test A passes at 1e-10 for every listed topology and delta schedule.
- Test B's checks pass, with sigma_2(W) logged.
- Test C runs and its `rho` is recorded above — whatever it says.

In [ ]:
print(f"Test A (mass conservation): {'PASS' if result_a.passed else 'FAIL'}")
print(f"Test B (spreading, ring):   {'PASS' if result_ring.residual_bound_satisfied else 'FAIL'}")
print(f"Test C (closed-loop diagnostic): RAN, rho={result_c.rho:.4f} (recorded, not a pass/fail gate)")